# MOSS 143-parameter forward model interface

This notebook is designed for GSA/OAT work. The grouped parameter table is loaded before the executable model definition; all model kernels and helper functions are kept in the notebook for reproducibility.

In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import math
import re
import sys
import time
import warnings

try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "outputs" / "oat_143" / "moss_143_parameter_ranges_oat.xlsx").exists():
    cwd_root = Path.cwd().resolve()
    if (cwd_root / "outputs" / "oat_143" / "moss_143_parameter_ranges_oat.xlsx").exists():
        PROJECT_ROOT = cwd_root
LOCAL_DEPS = PROJECT_ROOT / ".deps_moss"
if LOCAL_DEPS.exists():
    sys.path.insert(0, str(LOCAL_DEPS))

import numpy as np
import pandas as pd

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except Exception:
    plt = None

PARAMETER_TABLE_PATH = PROJECT_ROOT / "outputs" / "oat_143" / "moss_143_parameter_ranges_oat.xlsx"
PARAMETER_TABLE_CSV_PATH = PARAMETER_TABLE_PATH.with_suffix(".csv")
PARAMETER_SHEET = "Parameter_Ranges_143"
MODEL_KEY = "baseline"
PRIMARY_OUTPUT = "GFR"
DEFAULT_POINTS_PER_PARAMETER = 10
PRESSURE_STEP = 5
PRESSURES = np.arange(180.0, 89.0, -PRESSURE_STEP)

OAT_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "oat_143"
FIGURE_DIR = OAT_OUTPUT_DIR / "figures"
BY_PARAMETER_FIGURE_DIR = FIGURE_DIR / "by_parameter"
for path in [OAT_OUTPUT_DIR, FIGURE_DIR, BY_PARAMETER_FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Parameter table:", PARAMETER_TABLE_PATH)
print("Parameter table CSV fallback:", PARAMETER_TABLE_CSV_PATH)


def read_parameter_table_file(path=PARAMETER_TABLE_PATH, sheet_name=PARAMETER_SHEET):
    path = Path(path)
    csv_path = path.with_suffix(".csv")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    try:
        return pd.read_excel(path, sheet_name=sheet_name)
    except ImportError as exc:
        if csv_path.exists() and "openpyxl" in str(exc):
            warnings.warn(f"openpyxl is unavailable; reading {csv_path} instead of {path}")
            return pd.read_csv(csv_path)
        raise
    except FileNotFoundError:
        if csv_path.exists():
            warnings.warn(f"{path} not found; reading {csv_path} instead")
            return pd.read_csv(csv_path)
        raise



In [ ]:
GSA_PARAMETER_TABLE_PREVIEW = read_parameter_table_file(PARAMETER_TABLE_PATH, sheet_name=PARAMETER_SHEET)
assert len(GSA_PARAMETER_TABLE_PREVIEW) == 143

PARAMETER_GROUP_IDS_143 = GSA_PARAMETER_TABLE_PREVIEW["group_id"].tolist()
PARAMETER_BASELINE_143 = dict(zip(GSA_PARAMETER_TABLE_PREVIEW["group_id"], GSA_PARAMETER_TABLE_PREVIEW["baseline"]))
PARAMETER_RANGES_143 = {
    row.group_id: (float(row.lower_bound), float(row.upper_bound))
    for row in GSA_PARAMETER_TABLE_PREVIEW.itertuples(index=False)
}

print("143-parameter table loaded from:", PARAMETER_TABLE_PATH)
GSA_PARAMETER_TABLE_PREVIEW.head()

## Embedded baseline model

This cell is copied from `moss_model_complete.ipynb` without changing the scientific equations.

In [ ]:
EMBEDDED_MODELS = {'baseline': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 1.98]], [0.7142857142857143, ['Exp', [1.2, 4.2, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.00024000000000000003]], [0.35714285714285715, ['Const', 0.00048000000000000007]], [0.35714285714285715, ['Const', 0.00018000000000000004]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'adh_0': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.06]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.06]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.06]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.06]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.06]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.06]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 0.06]], [0.7142857142857143, ['Exp', [0.12, 1.98, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.00016]], [0.35714285714285715, ['Const', 0.00032]], [0.35714285714285715, ['Const', 0.00012]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'adh_25': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.54]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.54]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.54]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.54]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.54]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.54]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 0.54]], [0.7142857142857143, ['Exp', [0.39, 2.535, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.00018]], [0.35714285714285715, ['Const', 0.00036]], [0.35714285714285715, ['Const', 0.000135]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'adh_30': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.636]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.636]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.636]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.636]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.636]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.636]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 0.636]], [0.7142857142857143, ['Exp', [0.444, 2.646, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.000184]], [0.35714285714285715, ['Const', 0.000368]], [0.35714285714285715, ['Const', 0.000138]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'adh_35': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.732]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.732]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.732]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.732]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.732]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.732]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 0.732]], [0.7142857142857143, ['Exp', [0.498, 2.757, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.000188]], [0.35714285714285715, ['Const', 0.000376]], [0.35714285714285715, ['Const', 0.000141]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'adh_40': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.828]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.828]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.828]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.828]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.828]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.828]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 0.828]], [0.7142857142857143, ['Exp', [0.552, 2.868, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.000192]], [0.35714285714285715, ['Const', 0.000384]], [0.35714285714285715, ['Const', 0.000144]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'adh_45': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.924]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.924]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.924]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.924]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.924]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 0.924]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 0.924]], [0.7142857142857143, ['Exp', [0.606, 2.979, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.000196]], [0.35714285714285715, ['Const', 0.000392]], [0.35714285714285715, ['Const', 0.000147]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'adh_50': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.02]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.02]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.02]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.02]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.02]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.02]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0, 'pbm_const': 7.5, 'pbm_gfr': 0.13}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 1.02]], [0.7142857142857143, ['Exp', [0.66, 3.09, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.0002]], [0.35714285714285715, ['Const', 0.0004]], [0.35714285714285715, ['Const', 0.00015]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'adh_100': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 1.98]], [0.7142857142857143, ['Exp', [1.2, 4.2, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.00024000000000000003]], [0.35714285714285715, ['Const', 0.00048000000000000007]], [0.35714285714285715, ['Const', 0.00018000000000000004]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'eff_1': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 1.98]], [0.7142857142857143, ['Exp', [1.2, 4.2, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.00024000000000000003]], [0.35714285714285715, ['Const', 0.00048000000000000007]], [0.35714285714285715, ['Const', 0.00018000000000000004]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'eff_2': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 1.98]], [0.7142857142857143, ['Exp', [1.2, 4.2, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.00024000000000000003]], [0.35714285714285715, ['Const', 0.00048000000000000007]], [0.35714285714285715, ['Const', 0.00018000000000000004]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'eff_3': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 1.98]], [0.7142857142857143, ['Exp', [1.2, 4.2, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.00024000000000000003]], [0.35714285714285715, ['Const', 0.00048000000000000007]], [0.35714285714285715, ['Const', 0.00018000000000000004]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'eff_4': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 1.98]], [0.7142857142857143, ['Exp', [1.2, 4.2, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.00024000000000000003]], [0.35714285714285715, ['Const', 0.00048000000000000007]], [0.35714285714285715, ['Const', 0.00018000000000000004]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}, 'eff_5': {'r_renal_art': 0.0025, 'p_art': 140.0, 'p_venous': 4.0, 'hct': 0.45, 'c_protein': 5.5, 'c_urea': 5.0, 'c_k': 0.0, 'c_na': 135.0, 'c_nrs': 0.33, 'neg_myo_desc': False, 'pct_decr_fr': ['Some', 0.009], 'nephrons': [{'r_basal': 0.1, 'r_glom': 0.008, 'r_eff': 0.15, 'p_myo_refn': 100.0, 'kf': 2.5, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.3, 'tgf_min_na': 20.0, 'tgf_max_na': 110.0, 'tgf_response': 0.0018, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': False, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[1.0, ['Const', 13.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.0009000000000000002]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00444]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.2, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7000000000000001, ['Const', 0.001554]], [0.3, ['Const', 0.00063]]], [[0.7000000000000001, ['Const', 70.0]], [0.3, ['Const', 70.0]]], [[0.7000000000000001, ['Const', 0.0006600000000000001]], [0.3, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7000000000000001, ['Const', 0.00054]], [0.3, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': ['Some', {'length': 0.1, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[1.0, ['Const', 0.00063]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}], 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 10.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.8371703641691084, ['Const', 13.8]], [0.1628296358308916, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.8371703641691084, ['Const', 0.00030000000000000003]], [0.1628296358308916, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.23890000000000006, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 0.001554]], [0.25115110925073253, ['Const', 0.00063]]], [[0.1628296358308916, ['Const', 0.0]], [0.5860192549183759, ['Const', 70.0]], [0.25115110925073253, ['Const', 70.0]]], [[0.1628296358308916, ['Const', 0.04800000000000001]], [0.5860192549183759, ['Const', 0.0006600000000000001]], [0.25115110925073253, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.1628296358308916, ['Const', 0.114]], [0.5860192549183759, ['Const', 0.0003600000000000001]], [0.25115110925073253, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.6990562740300594, ['Const', 13.8]], [0.3009437259699406, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.6990562740300594, ['Const', 0.00030000000000000003]], [0.3009437259699406, ['Const', 0.12000000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.28609999999999997, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 0.001554]], [0.2097168822090178, ['Const', 0.00063]]], [[0.30094372596994057, ['Const', 0.0]], [0.4893393918210415, ['Const', 70.0]], [0.2097168822090178, ['Const', 70.0]]], [[0.30094372596994057, ['Const', 0.04800000000000001]], [0.4893393918210415, ['Const', 0.0006600000000000001]], [0.2097168822090178, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.30094372596994057, ['Const', 0.114]], [0.4893393918210415, ['Const', 0.0003600000000000001]], [0.2097168822090178, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5747126436781609, ['Const', 13.8]], [0.17011494252873563, ['Const', 13.2]], [0.25517241379310346, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.5747126436781609, ['Const', 0.00030000000000000003]], [0.17011494252873563, ['Const', 0.0078]], [0.25517241379310346, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.348, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 0.001554]], [0.1724137931034483, ['Const', 0.00063]]], [[0.4252873563218391, ['Const', 0.0]], [0.40229885057471265, ['Const', 70.0]], [0.1724137931034483, ['Const', 70.0]]], [[0.4252873563218391, ['Const', 0.04800000000000001]], [0.40229885057471265, ['Const', 0.0006600000000000001]], [0.1724137931034483, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.4252873563218391, ['Const', 0.114]], [0.40229885057471265, ['Const', 0.0003600000000000001]], [0.1724137931034483, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.45248868778280543, ['Const', 13.8]], [0.21900452488687783, ['Const', 13.2]], [0.32850678733031674, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.45248868778280543, ['Const', 0.00030000000000000003]], [0.21900452488687783, ['Const', 0.0078]], [0.32850678733031674, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.442, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 0.001554]], [0.13574660633484165, ['Const', 0.00063]]], [[0.5475113122171946, ['Const', 0.0]], [0.31674208144796384, ['Const', 70.0]], [0.13574660633484165, ['Const', 70.0]]], [[0.5475113122171946, ['Const', 0.04800000000000001]], [0.31674208144796384, ['Const', 0.0006600000000000001]], [0.13574660633484165, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.5475113122171946, ['Const', 0.114]], [0.31674208144796384, ['Const', 0.0003600000000000001]], [0.13574660633484165, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}, {'r_basal': 0.05, 'r_glom': 0.008, 'r_eff': 0.075, 'p_myo_refn': 100.0, 'kf': 11.0, 'myo_asc_scale': 0.0, 'myo_desc_scale': 0.5, 'tgf_min_na': 20.0, 'tgf_max_na': 135.0, 'tgf_response': 0.0011, 'pbm_const': 7.5, 'pbm_gfr': 0.13, 'to_dvr': True, 'pct_vol_frac': 0.6, 'pct_na_frac': 0.6, 'pct_urea_frac': 0.4, 'dlh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.2857142857142857, ['Const', 13.8]], [0.2857142857142857, ['Const', 13.2]], [0.42857142857142855, ['Const', 0.0]]]], 't_na': 'NoTransport', 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.00030000000000000003]], [0.2857142857142857, ['Const', 0.0078]], [0.42857142857142855, ['Const', 0.10800000000000001]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'alh': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': 'NoTransport', 't_na': ['Active', [[[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 0.001554]], [0.08571428571428572, ['Const', 0.00063]]], [[0.7142857142857143, ['Const', 0.0]], [0.19999999999999998, ['Const', 70.0]], [0.08571428571428572, ['Const', 70.0]]], [[0.7142857142857143, ['Const', 0.04800000000000001]], [0.19999999999999998, ['Const', 0.0006600000000000001]], [0.08571428571428572, ['Const', 0.0006600000000000001]]]]], 't_urea': ['Permeable', [[0.7142857142857143, ['Const', 0.114]], [0.19999999999999998, ['Const', 0.0003600000000000001]], [0.08571428571428572, ['Const', 0.00084]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_alh': 'None', 'dct': {'length': 0.25, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.5, ['Const', 0.0]], [0.5, ['Const', 1.98]]]], 't_na': ['Active', [[[1.0, ['Const', 0.0005400000000000001]]], [[1.0, ['Const', 70.0]]], [[1.0, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[1.0, ['Const', 0.00018000000000000004]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cort_dx': 0.01, 'count': 1.0}], 'med_c_nrs': [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], 'dvr': {'length': 0.7, 'radius': [[1.0, ['Const', 0.001]]], 't_vol': ['Permeable', [[0.08571428571428572, ['Const', 2.4]], [0.20000000000000004, ['Const', 0.6]], [0.7142857142857143, ['Const', 1.2]]]], 't_na': ['Permeable', [[1.0, ['Const', 0.04800000000000001]]]], 't_urea': ['Permeable', [[0.0857142857142857, ['Const', 0.024000000000000004]], [0.2, ['Const', 0.04800000000000001]], [0.5714285714285714, ['Const', 0.12000000000000001]], [0.14285714285714285, ['Const', 0.24000000000000002]]]], 'r_na': [[1.0, ['Const', 0.5]]], 'r_urea': [[1.0, ['Const', 0.5]]], 'r_nrs': [[1.0, ['Const', 0.0]]]}, 'dvr_count': [[0.08571428571428572, ['Const', 30.0]], [0.19999999999999998, ['Linear', [30.0, 9.9]]], [0.05557142857142857, ['Const', 9.9]], [0.06742857142857142, ['Const', 7.920000000000001]], [0.08842857142857143, ['Const', 5.94]], [0.13428571428571429, ['Const', 3.9600000000000004]], [0.3685714285714286, ['Const', 1.9800000000000002]]], 'coll_duct': {'length': 0.7, 'radius': [[1.0, ['Exp', [0.0037499999999999994, 0.00015000000000000001, -3.0]]]], 't_vol': ['Permeable', [[0.28571428571428575, ['Const', 1.98]], [0.7142857142857143, ['Exp', [1.2, 4.2, -3.0]]]]], 't_na': ['Active', [[[0.28571428571428575, ['Const', 0.00024000000000000003]], [0.35714285714285715, ['Const', 0.00048000000000000007]], [0.35714285714285715, ['Const', 0.00018000000000000004]]], [[0.28571428571428575, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]], [0.35714285714285715, ['Const', 70.0]]], [[0.28571428571428575, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]], [0.35714285714285715, ['Const', 0.0006000000000000001]]]]], 't_urea': ['Permeable', [[0.2857142857142857, ['Const', 0.0021]], [0.5714285714285714, ['Const', 0.0021]], [0.14285714285714285, ['Const', 0.06600000000000002]]]], 'r_na': [[1.0, ['Const', 1.0]]], 'r_urea': [[1.0, ['Const', 1.0]]], 'r_nrs': [[1.0, ['Const', 1.0]]]}, 'cd_k': ['Some', 20.0]}}
print("Embedded models:", list(EMBEDDED_MODELS))


## Exact numerical and physiological kernels

Copied from the source notebook.

In [ ]:
from __future__ import annotations
import json, math, re, time
from pathlib import Path
from dataclasses import dataclass
import numpy as np
from scipy.optimize import root
from numba import njit

MOLAR_H2O = (1e3 * (1.0 - (37.0-4.0)**2/(119000.0+1365.0*37.0-4.0*37.0**2))) / 18.01528

# Flow vector: rate(nL/min), Na, urea, K, NRS (mmol/min), protein(g/min)
def flow(rate, c_na=0., c_urea=0., c_k=0., c_nrs=0., c_p=0.):
    q=float(rate); return np.array([q,c_na*q*1e-9,c_urea*q*1e-9,c_k*q*1e-9,c_nrs*q*1e-9,10*c_p*q*1e-9],float)
def empty(): return flow(0.)
def conc(f, ix): return abs(f[ix]/(f[0]*1e-9))
def c_na(f): return conc(f,1)
def c_urea(f): return conc(f,2)
def c_k(f): return conc(f,3)
def c_nrs(f): return conc(f,4)
def c_protein(f): return conc(f,5)/10.
def mosm(f): return 1.82*(c_na(f)+c_k(f))+c_urea(f)+c_nrs(f)
def oncotic_c(c): return 2.1*c+.16*c*c+.009*c*c*c
def mean_concs(a,b):
    q=.5*(a[0]+b[0]); return flow(q,.5*(c_na(a)+c_na(b)),.5*(c_urea(a)+c_urea(b)),.5*(c_k(a)+c_k(b)),.5*(c_nrs(a)+c_nrs(b)),.5*(c_protein(a)+c_protein(b)))
def flow_from_istm(istm,rate,incl_nrs=False): return flow(rate,istm[0],istm[1],istm[2],istm[3] if incl_nrs else 0.)

def value_at(value, rel_x):
    start=0.; rel_x=float(rel_x)
    for i,(length,rg) in enumerate(value):
        nxt=start+length
        if nxt>rel_x or i==len(value)-1:
            z=np.clip((rel_x-start)/length,0.,1.) if nxt>rel_x else 1.
            tag=rg[0]; arg=rg[1]
            if tag=='Const': return float(arg)
            if tag=='Linear': return arg[0]+z*(arg[1]-arg[0])
            if tag=='Exp':
                a,b,e=arg; return a+(b-a)*np.expm1(e*z)/np.expm1(e)
            raise ValueError(tag)
        start=nxt
    raise RuntimeError('value_at')

def active_of(tx, x):
    if tx=='NoTransport': return 0.,0.,0.
    if tx[0]=='Permeable': return 0.,0.,value_at(tx[1],x)
    if tx[0]=='Active':
        vm,km,p=tx[1]; return value_at(vm,x),value_at(km,x),value_at(p,x)
    raise ValueError(tx)

def flux_between(tube,dx,rel_x,filtrate,istm):
    r=value_at(tube['radius'],rel_x); surface=2*np.pi*r*dx
    pvol=active_of(tube['t_vol'],rel_x)[2]
    vmax,km,pna=active_of(tube['t_na'],rel_x)
    pur=active_of(tube['t_urea'],rel_x)[2]
    rna=value_at(tube['r_na'],rel_x); rur=value_at(tube['r_urea'],rel_x); rnrs=value_at(tube['r_nrs'],rel_x)
    jna=(c_na(filtrate)-istm[0])*1e-3*surface*pna
    jur=(c_urea(filtrate)-istm[1])*1e-3*surface*pur
    grad=1.82*rna*(istm[0]-c_na(filtrate))+1.82*(istm[2]-c_k(filtrate))+rur*(istm[1]-c_urea(filtrate))+rnrs*(istm[3]-c_nrs(filtrate))
    jv_mmol=grad*1e-3*surface*pvol
    cn=c_na(filtrate); jna += surface*vmax*cn/(cn+km)
    jv=1e6*jv_mmol/MOLAR_H2O
    jna += (1-rna)*jv*1e-9*.5*(cn+istm[0])
    jur += (1-rur)*jv*1e-9*.5*(c_urea(filtrate)+istm[1])
    return np.array([jv,jna,jur,0.,0.,0.])

def numerical_jac(fn,x,dx):
    x=np.asarray(x,float).copy(); y0=np.asarray(fn(x),float); J=np.empty((len(y0),len(x)))
    for i in range(len(x)):
        xi=x[i]; x[i]=xi+dx; J[:,i]=(np.asarray(fn(x))-y0)/dx; x[i]=xi
    return J

def newton_solve(fn,x0,dx=1e-4,tol=1e-6,maxiter=10,jac=None):
    x=np.asarray(x0,float).copy(); xs=[]
    for count in range(maxiter+1):
        y=np.asarray(fn(x),float)
        if np.all(np.abs(y)<tol): return x,y,count
        if count>=maxiter: raise RuntimeError(f'Newton no solution: maxerr={np.max(np.abs(y)):.3e}')
        J=jac(x) if jac else numerical_jac(fn,x,dx)
        xn=x-np.linalg.solve(J,y)
        if len(xs)>=2:
            x1,x2=xs[-1],xs[-2]; d1=xn-x1; d2=x2-x1
            rd=np.empty_like(d1)
            both_zero=(d1==0)&(d2==0); denom_zero=(d2==0)&~both_zero; regular=~(both_zero|denom_zero)
            rd[both_zero]=0.; rd[denom_zero]=np.abs(d1[denom_zero]); rd[regular]=np.abs(1-d1[regular]/d2[regular])
            ck=(d1==0)|((rd<1e-2)&(d1>1e2*rd))
            if np.all(ck): xn=.5*(xn+x1)
        xs.append(x.copy()); x=xn
    raise AssertionError

def flux_step(tube,dx,rel_x,istm,inflow,tol=1e-9):
    if inflow[0]<=0: raise ValueError('negative inflow')
    guess=np.array([inflow[0],c_na(inflow),c_urea(inflow),c_k(inflow),c_nrs(inflow)])
    def evaluate(g,output=False):
        gf=flow(g[0],g[1],g[2],g[3],g[4]); fx=flux_between(tube,dx,rel_x,gf,istm); out=inflow-fx; mean=mean_concs(inflow,out)
        err=np.array([mean[0]-g[0],c_na(mean)-g[1],c_urea(mean)-g[2],c_k(mean)-g[3],c_nrs(mean)-g[4]])
        return (err,out,fx) if output else err
    sol,_,_=newton_solve(evaluate,guess,dx=1e-6,tol=tol,maxiter=30)
    _,out,fx=evaluate(sol,True)
    if out[0]<=0: raise ValueError('negative outflow')
    return out,fx

def flux_profile(tube,dx,istm_fn,inflow):
    # OCaml Util.--. advances by repeated floating-point addition. Retaining
    # that detail matters at piecewise-transport boundaries (e.g. DCT x=0.5).
    f=inflow.copy(); flows=[f.copy()]; fluxes=[]; total=empty(); mid=.5*dx
    while mid <= tube['length'] + 1e-9*dx:
        f,fx=flux_step(tube,dx,mid/tube['length'],istm_fn(mid),f); total+=fx; flows.append(f.copy()); fluxes.append(fx.copy()); mid+=dx
    return f,total,np.asarray(flows),np.asarray(fluxes)

def load_json(path):
    txt=Path(path).read_text(encoding='utf-8'); txt=re.sub(r'/\*.*?\*/','',txt,flags=re.S); return json.loads(txt)

@dataclass
class MedOutput:
    istm: np.ndarray; cd_out: np.ndarray; avr_out: np.ndarray; md_flow: list; dct_out: list
    net_dct_flux: np.ndarray; dlh_f: list; dlh_j: list; alh_f: list; alh_j: list
    dvr_f: list; dvr_j: list; dvr_s: list; dvri_f: list; dvri_j: list
    dct_f: list; dct_j: list; cd_f: np.ndarray; cd_j: np.ndarray; avr: np.ndarray
    net_flow: np.ndarray; net_flux: np.ndarray; net_shunt: np.ndarray

def med_guess(model,inflows,om_na=3.,om_ur=3.,im_na=4.,im_ur=5.):
    ns=len(model['med_c_nrs'])-1; depth=model['dvr']['length']; om_depth=min(n['dlh']['length'] for n in model['nephrons']); im_depth=depth-om_depth
    dna=c_na(inflows[0]); dur=c_urea(inflows[0]); g=np.zeros(2*ns)
    for ix in range(ns):
        frac=ix/(ns-1)
        if frac<=om_depth/depth:
            z=frac*depth/om_depth; g[2*ix]=(1+(om_na-1)*z)*dna; g[2*ix+1]=(1+(om_ur-1)*z)*dur
        else:
            z=(frac*depth-om_depth)/im_depth; g[2*ix]=(om_na+(im_na-om_na)*z)*dna; g[2*ix+1]=(om_ur+(im_ur-om_ur)*z)*dur
    return g

def med_output_of(model,inflows,vec):
    ns=len(model['med_c_nrs'])-1; depth=model['dvr']['length']; dx=depth/ns
    cnrs=.5*(np.asarray(model['med_c_nrs'][:-1])+np.asarray(model['med_c_nrs'][1:]))
    istm=np.column_stack((vec[0::2],vec[1::2],np.zeros(ns),cnrs))
    nl=len(model['nephrons']); nd=1
    net_flow=np.zeros((ns+1,6)); net_flux=np.zeros((ns,6)); net_shunt=np.zeros((ns,6))
    dvr_f=[]; dvr_j=[]; dvr_s=[]; dvri_f=[]; dvri_j=[]
    # Aggregate DVR
    tube=model['dvr']; counts=model['dvr_count']; nf=inflows[0].copy(); flows=[nf.copy()]; fxs=[]; shs=[]; ifl=[nf/value_at(counts,0.)]; ifx=[]
    net_flow[0]+=nf
    for s in range(ns):
        rel=(s+.5)/ns; nin=value_at(counts,s/ns); nout=value_at(counts,(s+1)/ns)
        if np.isclose(nin,nout): sh=empty()
        else:
            sh=nf*(1-nout/nin); nf=nf*(nout/nin)
        if np.isclose(nout,0): ind=empty(); out=empty(); fx=empty()
        else:
            ind=nf/nout; out,fx=flux_step(tube,dx,rel,istm[s],ind)
        nf=out*nout; nfx=fx*nout
        flows.append(nf.copy()); fxs.append(nfx.copy()); shs.append(sh.copy()); ifl.append(out.copy()); ifx.append(fx.copy())
        net_flow[s+1]+=nf; net_flux[s]+=nfx; net_shunt[s]+=sh
    dvr_f.append(np.asarray(flows)); dvr_j.append(np.asarray(fxs)); dvr_s.append(np.asarray(shs)); dvri_f.append(np.asarray(ifl)); dvri_j.append(np.asarray(ifx))

    def tube_out(tube,num,inflow,slice0=0,desc=True):
        n=int(round(tube['length']/dx)); s=slice0; f=inflow.copy(); fs=[f.copy()]; js=[]
        if desc: net_flow[s]+=f*num
        else: net_flow[s+1]-=f*num
        for k in range(n):
            out,fx=flux_step(tube,dx,(k+.5)/n,istm[s],f); so=out*num; sj=fx*num
            if desc: net_flow[s+1]+=so
            else: net_flow[s]-=so
            net_flux[s]+=sj; fs.append(out.copy()); js.append(fx.copy()); f=out; s=s+1 if desc else s-1
        return s,f,np.asarray(fs),np.asarray(js)

    dlh_f=[]; dlh_j=[]; alh_f=[]; alh_j=[]; dct_f=[]; dct_j=[]; md_flow=[]; dct_out=[]
    cd_in=empty(); dct_net=empty(); cortex=np.array([model['c_na'],model['c_urea'],model['c_k'],model['c_nrs']])
    for i,nph in enumerate(model['nephrons']):
        s,turn,df,dj=tube_out(nph['dlh'],nph['count'],inflows[i+1],0,True)
        s2,aout,af,aj=tube_out(nph['alh'],nph['count'],turn,s-1,False)
        if nph['cort_alh'][0]=='Some':
            md,ef,efs,efx=flux_profile(nph['cort_alh'][1],nph['cort_dx'],lambda _:cortex,aout)
        else: md=aout; ef=empty(); efs=np.empty((0,6)); efx=np.empty((0,6))
        dout,lf,lfs,lfx=flux_profile(nph['dct'],nph['cort_dx'],lambda _:cortex,md)
        if len(efs): lfs=lfs[1:]
        fs=np.concatenate((efs,lfs)) if len(efs) else lfs; js=np.concatenate((efx,lfx)) if len(efx) else lfx
        dlh_f.append(df); dlh_j.append(dj); alh_f.append(af); alh_j.append(aj); dct_f.append(fs); dct_j.append(js); md_flow.append(md); dct_out.append(dout)
        cd_in+=dout*nph['count']; dct_net+=(ef+lf)*nph['count']
    if model['cd_k'][0]=='Some': cd_in=flow(cd_in[0],c_na(cd_in),c_urea(cd_in),model['cd_k'][1],c_nrs(cd_in))
    _,cd_out,cdf,cdj=tube_out(model['coll_duct'],1.,cd_in,0,True)
    # Ascending vasa recta
    avr_in=dvr_f[0][-1].copy(); net_flow[ns]-=avr_in; afs=[avr_in.copy()]; f=avr_in
    combined=net_flux+net_shunt
    for s in range(ns-1,-1,-1):
        rate=f[0]+combined[s,0]; f=flow_from_istm(istm[s],rate,True); net_flow[s]-=f; afs.append(f.copy())
    avr=np.asarray(afs[::-1]); avr_out=f
    errs=np.empty(2*ns)
    for s in range(ns): errs[2*s]=1e9*(cd_out[1]-net_flow[s,1]); errs[2*s+1]=1e9*(cd_out[2]-net_flow[s,2])
    out=MedOutput(istm,cd_out,avr_out,md_flow,dct_out,dct_net,dlh_f,dlh_j,alh_f,alh_j,dvr_f,dvr_j,dvr_s,dvri_f,dvri_j,dct_f,dct_j,cdf,cdj,avr,net_flow,net_flux,net_shunt)
    return errs,out

def gfr_of_nephrons(model):
    n=len(model['nephrons']); return np.r_[np.full(n,model['p_art']/6.),np.full(n,model['p_art']*.5)]

def myogenic_from_tgf(model,rtgf,p_i,i):
    n=model['nephrons'][i]; rb=n['r_basal']; rg=n['r_glom']; re=n['r_eff']; ga=n['myo_asc_scale']; gd=n['myo_desc_scale']; po=n['p_myo_refn']
    pi=p_i if model['neg_myo_desc'] or p_i>po else po
    t1=ga*rtgf[i]/(rg+re); t2=gd*(pi/po-1)
    rma=t1*(rb+t2*(rb+rtgf[i]+rg+re))/(1-t1*t2); rmd=t2*(rb+rma+rtgf[i]+rg+re)
    return rma,rmd

def solve_gfr(model,rtgf):
    n=len(model['nephrons'])
    def eval_pi(pi):
        myo=np.array([myogenic_from_tgf(model,rtgf,pi,i) for i in range(n)])
        raff=np.array([model['nephrons'][i]['r_basal']+rtgf[i]+myo[i,0]+myo[i,1] for i in range(n)])
        def residual(x):
            gfr=x[:n]; pg0=x[n:]; counts=np.array([z['count'] for z in model['nephrons']])
            pnew=(model['p_art']/model['r_renal_art']+np.sum(counts*pg0/raff))/(1/model['r_renal_art']+np.sum(counts/raff))
            bf=(pnew-pg0)/raff; es=[]
            for i,z in enumerate(model['nephrons']):
                pf=bf[i]*(1-model['hct']); plasma=pf; press=pg0[i]; pb=z['pbm_const']+z['pbm_gfr']*gfr[i]
                # Util.(0.0, 1e-3) --. 1.0 is inclusive: 1001 Euler updates.
                for _ in range(1001):
                    cp=model['c_protein']*pf/plasma
                    dq=-z['kf']*(press-oncotic_c(cp)-pb); dp=-z['r_glom']*plasma
                    plasma+=.001*dq; press+=.001*dp
                es.extend([gfr[i]-(pf-plasma), pnew-bf[i]*raff[i]-(pg0[i]-press)-(bf[i]-gfr[i])*z['r_eff']-model['p_venous']])
            return np.asarray(es),pnew,bf,myo,raff
        sol=root(lambda x:residual(x)[0],gfr_of_nephrons(model),method='hybr',options={'xtol':1e-9})
        er,pnew,bf,myo,raff=residual(sol.x); return pi-pnew,sol.x,pnew,bf,myo,raff,np.max(np.abs(er))
    # Scalar p_i Newton, with nested GFR solve, matching source dependency.
    pi=model['p_art']; last=None
    for _ in range(10):
        e,*last=eval_pi(pi)
        if abs(e)<1e-9: break
        e2=eval_pi(pi+1e-4)[0]; pi-=e/((e2-e)/1e-4)
    x,pnew,bf,myo,raff,gerr=last; return {'gfr':x[:n],'pg0':x[n:],'p_int':pnew,'bf':bf,'myo':myo,'raff':raff,'gerr':gerr}

def plasma_of(model,rate,c_p=0.): return flow(rate,model['c_na'],model['c_urea'],model['c_k'],model['c_nrs'],c_p)

def prepare_medulla_inflows(model,vascular):
    nph=model['nephrons']; gfr=vascular['gfr']; bf=vascular['bf']; ctx=empty(); dvr=empty(); gflows=[]
    for i,n in enumerate(nph):
        pf=bf[i]*(1-model['hct']); cp=model['c_protein']/(1-gfr[i]/pf); eff=plasma_of(model,pf-gfr[i],cp)*n['count']
        if n['to_dvr']: dvr+=eff
        else: ctx+=eff
        gflows.append(plasma_of(model,gfr[i]))
    scale=1.
    if model['pct_decr_fr'][0]=='Some' and model['p_art']>100: scale=1-(model['p_art']-100)*model['pct_decr_fr'][1]
    pct=[]
    for gf,n in zip(gflows,nph):
        fx=np.array([gf[0]*n['pct_vol_frac']*scale,gf[1]*n['pct_na_frac']*scale,gf[2]*n['pct_urea_frac']*scale,0,0,0.])
        out=gf-fx; pct.append(out); ctx+=fx*n['count']
    return [dvr]+pct,ctx,gflows

def solve_state_given_tgf(model,rtgf,med_vec=None,med_method='hybr'):
    vascular=solve_gfr(model,np.asarray(rtgf)); inflows,ctx,gflows=prepare_medulla_inflows(model,vascular)
    x0=med_guess(model,inflows) if med_vec is None else np.asarray(med_vec)
    calls=0; last=None
    def fn(x):
        nonlocal calls,last; calls+=1; er,last=med_output_of(model,inflows,x); return er
    if med_method=='hybr':
        sol=root(fn,x0,method='hybr',options={'xtol':1e-8,'maxfev':5000}); x=sol.x
    else:
        # Log coordinates preserve positive interstitial concentrations while
        # allowing matrix-free/Broyden acceleration of the exact residual.
        def fnlog(z): return fn(np.exp(np.clip(z,-5.,9.)))
        sol=root(fnlog,np.log(x0),method=med_method,options={'fatol':1e-6,'maxiter':500}); x=np.exp(np.clip(sol.x,-5.,9.))
    er=fn(x); return {'vascular':vascular,'med_soln':x,'med_out':last,'med_error':np.max(np.abs(er)),'med_calls':calls,'med_success':sol.success,'inflows':inflows,'ctx':ctx,'gflows':gflows}

# ---------- Numba-accelerated exact medullary residual ----------
MAX_SEG=8; NFIELD=9
def _enc_value(v):
    lens=np.zeros(MAX_SEG); typ=np.zeros(MAX_SEG,np.int64); arg=np.zeros((MAX_SEG,3)); n=len(v)
    for i,(ln,rg) in enumerate(v):
        lens[i]=ln; tag=rg[0]; a=rg[1]
        if tag=='Const': typ[i]=0; arg[i,0]=a
        elif tag=='Linear': typ[i]=1; arg[i,:2]=a
        elif tag=='Exp': typ[i]=2; arg[i,:3]=a
    return n,lens,typ,arg
def encode_model(model):
    tubes=[]; has_c=[]
    for n in model['nephrons']:
        tubes.extend([n['dlh'],n['alh']]);
        if n['cort_alh'][0]=='Some': tubes.append(n['cort_alh'][1]); has_c.append(1)
        else:
            tubes.append({'length':0.,'radius':[[1.,['Const',0.]]],'t_vol':'NoTransport','t_na':'NoTransport','t_urea':'NoTransport','r_na':[[1.,['Const',1.]]],'r_urea':[[1.,['Const',1.]]],'r_nrs':[[1.,['Const',1.]]]}); has_c.append(0)
        tubes.append(n['dct'])
    dvr_id=len(tubes); tubes.append(model['dvr']); cd_id=len(tubes); tubes.append(model['coll_duct'])
    nt=len(tubes); lengths=np.zeros(nt); nseg=np.zeros((nt,NFIELD),np.int64); lens=np.zeros((nt,NFIELD,MAX_SEG)); typ=np.zeros((nt,NFIELD,MAX_SEG),np.int64); args=np.zeros((nt,NFIELD,MAX_SEG,3))
    for ti,t in enumerate(tubes):
        lengths[ti]=t['length']; fields=[t['radius']]
        for tx,which in [(t['t_vol'],2),(t['t_na'],0),(t['t_na'],1),(t['t_na'],2),(t['t_urea'],2)]:
            if tx=='NoTransport': val=[[1.,['Const',0.]]]
            elif tx[0]=='Permeable': val=tx[1] if which==2 else [[1.,['Const',0.]]]
            else: val=tx[1][which]
            fields.append(val)
        fields += [t['r_na'],t['r_urea'],t['r_nrs']]
        for fi,v in enumerate(fields):
            nn,ll,tt,aa=_enc_value(v); nseg[ti,fi]=nn; lens[ti,fi]=ll; typ[ti,fi]=tt; args[ti,fi]=aa
    cn,cl,ct,ca=_enc_value(model['dvr_count'])
    counts=np.array([n['count'] for n in model['nephrons']]); cortdx=np.array([n['cort_dx'] for n in model['nephrons']])
    return (lengths,nseg,lens,typ,args,np.array(has_c,np.int64),counts,cortdx,dvr_id,cd_id,cn,cl,ct,ca,np.asarray(model['med_c_nrs']),float(model['c_na']),float(model['c_urea']),float(model['c_k']),float(model['c_nrs']),float(model['cd_k'][1] if model['cd_k'][0]=='Some' else -1.))

@njit(cache=False)
def _v_at(n,lens,typ,args,x):
    start=0.
    for i in range(n):
        nxt=start+lens[i]
        if nxt>x or i==n-1:
            z=(x-start)/lens[i] if nxt>x else 1.
            if z<0:z=0.
            if z>1:z=1.
            if typ[i]==0:return args[i,0]
            if typ[i]==1:return args[i,0]+z*(args[i,1]-args[i,0])
            return args[i,0]+(args[i,1]-args[i,0])*np.expm1(args[i,2]*z)/np.expm1(args[i,2])
        start=nxt
    return 0.
@njit(cache=False)
def _tv(ti,fi,x,nseg,lens,typ,args): return _v_at(nseg[ti,fi],lens[ti,fi],typ[ti,fi],args[ti,fi],x)
@njit(cache=False)
def _c(f,i): return abs(f[i]/(f[0]*1e-9))
@njit(cache=False)
def _mkflow(q,cna,cur,ck,cn): return np.array([q,cna*q*1e-9,cur*q*1e-9,ck*q*1e-9,cn*q*1e-9,0.])
@njit(cache=False)
def _between(ti,dx,x,g,ist,nseg,lens,typ,args):
    r=_tv(ti,0,x,nseg,lens,typ,args); area=2*np.pi*r*dx; pv=_tv(ti,1,x,nseg,lens,typ,args); vm=_tv(ti,2,x,nseg,lens,typ,args); km=_tv(ti,3,x,nseg,lens,typ,args); pna=_tv(ti,4,x,nseg,lens,typ,args); pur=_tv(ti,5,x,nseg,lens,typ,args); rna=_tv(ti,6,x,nseg,lens,typ,args); rur=_tv(ti,7,x,nseg,lens,typ,args); rnrs=_tv(ti,8,x,nseg,lens,typ,args)
    jna=(_c(g,1)-ist[0])*1e-3*area*pna; jur=(_c(g,2)-ist[1])*1e-3*area*pur
    grad=1.82*rna*(ist[0]-_c(g,1))+1.82*(ist[2]-_c(g,3))+rur*(ist[1]-_c(g,2))+rnrs*(ist[3]-_c(g,4))
    jv=1e6*(grad*1e-3*area*pv)/MOLAR_H2O; cn=_c(g,1); jna+=area*vm*cn/(cn+km)
    jna+=(1-rna)*jv*1e-9*.5*(cn+ist[0]); jur+=(1-rur)*jv*1e-9*.5*(_c(g,2)+ist[1])
    return np.array([jv,jna,jur,0.,0.,0.])
@njit(cache=False)
def _flux_eval(ti,dx,x,ist,fin,g,nseg,lens,typ,args):
    gf=_mkflow(g[0],g[1],g[2],g[3],g[4]); fx=_between(ti,dx,x,gf,ist,nseg,lens,typ,args); out=fin-fx; qm=.5*(fin[0]+out[0]); e=np.empty(5); e[0]=qm-g[0]
    for i in range(1,5): e[i]=.5*(_c(fin,i)+_c(out,i))-g[i]
    return e,out,fx
@njit(cache=False)
def _step(ti,dx,x,ist,fin,nseg,lens,typ,args):
    g=np.array([fin[0],_c(fin,1),_c(fin,2),_c(fin,3),_c(fin,4)])
    for it in range(30):
        e,out,fx=_flux_eval(ti,dx,x,ist,fin,g,nseg,lens,typ,args)
        if np.max(np.abs(e))<1e-9:return out,fx
        J=np.empty((5,5)); h=1e-6
        for j in range(5):
            gp=g.copy(); gp[j]+=h; ep,_,_=_flux_eval(ti,dx,x,ist,fin,gp,nseg,lens,typ,args); J[:,j]=(ep-e)/h
        g-=np.linalg.solve(J,e)
    return out,fx
@njit(cache=False)
def _profile(ti,dx,ist,fin,lengths,nseg,lens,typ,args):
    f=fin.copy(); total=np.zeros(6); mid=.5*dx
    while mid<=lengths[ti]+1e-9*dx:
        f,fx=_step(ti,dx,mid/lengths[ti],ist,f,nseg,lens,typ,args); total+=fx; mid+=dx
    return f,total

@njit(cache=False)
def med_residual_fast(vec,inflows,lengths,nseg,lens,typ,args,has_c,counts,cortdx,dvr_id,cd_id,dcn,dcl,dct,dca,mednrs,cort_na,cort_ur,cort_k,cort_nrs,cdk):
    ns=len(mednrs)-1; dx=lengths[dvr_id]/ns; ist=np.empty((ns,4))
    for s in range(ns): ist[s,0]=vec[2*s];ist[s,1]=vec[2*s+1];ist[s,2]=0.;ist[s,3]=.5*(mednrs[s]+mednrs[s+1])
    nf=np.zeros((ns+1,6)); nx=np.zeros((ns,6)); nsh=np.zeros((ns,6))
    # DVR
    f=inflows[0].copy(); nf[0]+=f
    for s in range(ns):
        nin=_v_at(dcn,dcl,dct,dca,s/ns); nout=_v_at(dcn,dcl,dct,dca,(s+1)/ns)
        sh=np.zeros(6)
        if abs(nin-nout)>1e-12: sh=f*(1-nout/nin); f*=nout/nin
        ind=f/nout; out,fx=_step(dvr_id,dx,(s+.5)/ns,ist[s],ind,nseg,lens,typ,args); f=out*nout; fx*=nout
        nf[s+1]+=f; nx[s]+=fx; nsh[s]+=sh
    avr_in=f.copy(); cdin=np.zeros(6); dctnet=np.zeros(6); cortex=np.array([cort_na,cort_ur,cort_k,cort_nrs])
    # Loops
    for i in range(6):
        fd=inflows[i+1].copy(); num=counts[i]; ti=4*i; nf[0]+=fd*num; n=int(.5+lengths[ti]/dx); sl=0
        for k in range(n):
            fd,fx=_step(ti,dx,(k+.5)/n,ist[sl],fd,nseg,lens,typ,args); nf[sl+1]+=fd*num; nx[sl]+=fx*num; sl+=1
        ti+=1; nf[sl]-=fd*num
        for k in range(n):
            sl-=1; fd,fx=_step(ti,dx,(k+.5)/n,ist[sl],fd,nseg,lens,typ,args); nf[sl]-=fd*num; nx[sl]+=fx*num
        ef=np.zeros(6)
        if has_c[i]==1: fd,ef=_profile(4*i+2,cortdx[i],cortex,fd,lengths,nseg,lens,typ,args)
        fd,lf=_profile(4*i+3,cortdx[i],cortex,fd,lengths,nseg,lens,typ,args); cdin+=fd*num; dctnet+=(ef+lf)*num
    if cdk>=0: cdin=_mkflow(cdin[0],_c(cdin,1),_c(cdin,2),cdk,_c(cdin,4))
    # CD
    f=cdin.copy(); nf[0]+=f
    for s in range(ns):
        f,fx=_step(cd_id,dx,(s+.5)/ns,ist[s],f,nseg,lens,typ,args); nf[s+1]+=f; nx[s]+=fx
    cdout=f.copy()
    # AVR
    f=avr_in.copy(); nf[ns]-=f
    for s in range(ns-1,-1,-1):
        rate=f[0]+nx[s,0]+nsh[s,0]; f=_mkflow(rate,ist[s,0],ist[s,1],ist[s,2],ist[s,3]); nf[s]-=f
    er=np.empty(2*ns)
    for s in range(ns): er[2*s]=1e9*(cdout[1]-nf[s,1]);er[2*s+1]=1e9*(cdout[2]-nf[s,2])
    return er

def fast_residual_fn(model,inflows):
    enc=encode_model(model); ina=np.asarray(inflows)
    def fn(x): return med_residual_fast(np.asarray(x),ina,*enc)
    return fn

def solve_gfr_joint(model,rtgf,guess=None):
    n=6
    if guess is None: z0=np.r_[gfr_of_nephrons(model),model['p_art']]
    else: z0=np.asarray(guess).copy()
    def residual(z):
        gfr=z[:n]; pg0=z[n:2*n]; pi=z[-1]
        myo=np.array([myogenic_from_tgf(model,rtgf,pi,i) for i in range(n)])
        raff=np.array([model['nephrons'][i]['r_basal']+rtgf[i]+myo[i,0]+myo[i,1] for i in range(n)])
        counts=np.array([x['count'] for x in model['nephrons']])
        pnew=(model['p_art']/model['r_renal_art']+np.sum(counts*pg0/raff))/(1/model['r_renal_art']+np.sum(counts/raff))
        bf=(pnew-pg0)/raff; es=[]
        for i,x in enumerate(model['nephrons']):
            pf=bf[i]*(1-model['hct']); plasma=pf; press=pg0[i]; pb=x['pbm_const']+x['pbm_gfr']*gfr[i]
            for _ in range(1001):
                cp=model['c_protein']*pf/plasma; plasma+=.001*(-x['kf']*(press-oncotic_c(cp)-pb)); press+=.001*(-x['r_glom']*plasma)
            es.extend([gfr[i]-(pf-plasma),pnew-bf[i]*raff[i]-(pg0[i]-press)-(bf[i]-gfr[i])*x['r_eff']-model['p_venous']])
        es.append(pi-pnew); return np.asarray(es)
    sol=root(residual,z0,method='hybr',options={'xtol':1e-10}); z=sol.x
    gfr=z[:n];pg0=z[n:2*n];pi=z[-1];myo=np.array([myogenic_from_tgf(model,rtgf,pi,i) for i in range(n)]);raff=np.array([model['nephrons'][i]['r_basal']+rtgf[i]+myo[i,0]+myo[i,1] for i in range(n)]);bf=(pi-pg0)/raff
    return {'gfr':gfr,'pg0':pg0,'p_int':pi,'bf':bf,'myo':myo,'raff':raff,'gerr':np.max(np.abs(residual(z))),'z':z}

class FastMedullaSolver:
    def __init__(self): self.x=None; self.J=None; self.model_key=None
    def solve(self,model,inflows,guess=None):
        fn=fast_residual_fn(model,inflows)
        if guess is not None: x=np.asarray(guess).copy()
        elif self.x is not None and len(self.x)==2*(len(model['med_c_nrs'])-1): x=self.x.copy()
        else:
            # Independent pressure-aware cold guess; only affects convergence.
            sw=1/(1+np.exp((model['p_art']-115)/4)); x=med_guess(model,inflows,1.3+1.5*sw,1.15+1.8*sw,1.6+2.4*sw,1.3+3.7*sw)
        y=fn(x); calls=1
        # Reuse the previous exact Jacobian as a chord preconditioner.
        if self.J is not None:
            for _ in range(4):
                if np.max(np.abs(y))<1e-6: break
                try: xn=x-np.linalg.solve(self.J,y)
                except np.linalg.LinAlgError: break
                yn=fn(xn);calls+=1
                if np.max(np.abs(yn))>=np.max(np.abs(y))*1.2: break
                x,y=xn,yn
        it=0
        while np.max(np.abs(y))>=1e-6 and it<12:
            J=numerical_jac(fn,x,.1);calls+=len(x)+1; x=x-np.linalg.solve(J,y); y=fn(x);calls+=1;it+=1; self.J=J
        if np.max(np.abs(y))>=1e-6: raise RuntimeError(f'medulla failed {np.max(np.abs(y))}')
        self.x=x.copy(); return x,y,calls

class WholeKidneySolver:
    def __init__(self): self.med=FastMedullaSolver(); self.gfr_z=None; self.last_state=None; self.tgf_J=None
    def evaluate(self,model,rtgf,med_guess0=None):
        v=solve_gfr_joint(model,np.asarray(rtgf),self.gfr_z); self.gfr_z=v['z'].copy()
        inflows,ctx,gflows=prepare_medulla_inflows(model,v); mx,me,mcalls=self.med.solve(model,inflows,med_guess0)
        _,mo=med_output_of(model,inflows,mx)
        target=np.array([n['tgf_response']*(np.clip(c_na(mo.md_flow[i]),n['tgf_min_na'],n['tgf_max_na'])-n['tgf_min_na']) for i,n in enumerate(model['nephrons'])])
        err=np.asarray(rtgf)-target; st={'vascular':v,'med_soln':mx,'med_out':mo,'med_error':np.max(np.abs(me)),'med_calls':mcalls,'inflows':inflows,'ctx':ctx,'gflows':gflows,'rtgf':np.asarray(rtgf),'target_tgf':target}
        self.last_state=st; return err,st
    def solve(self,model,rtgf0=None,med_guess0=None):
        if rtgf0 is None:
            mn,mx=95.,160.; vals=[]
            for n in model['nephrons']:
                if model['p_art']<=mn: vals.append(.05)
                elif model['p_art']>=mx: vals.append(n['tgf_response']*(n['tgf_max_na']-n['tgf_min_na']))
                else: vals.append(.05+n['tgf_response']*(model['p_art']-mn)/(mx-mn)*(n['tgf_max_na']-n['tgf_min_na']))
            rtgf0=np.array(vals)
        first=[med_guess0]
        def fn(x): g=first[0];first[0]=None;return self.evaluate(model,x,g)[0]
        x=np.asarray(rtgf0).copy(); y=fn(x)
        # Chord continuation reuses the previous pressure's outer Jacobian.
        if self.tgf_J is not None:
            for _ in range(6):
                if np.max(np.abs(y))<1e-9: break
                xn=x-np.linalg.solve(self.tgf_J,y); yn=fn(xn)
                if np.max(np.abs(yn))>np.max(np.abs(y))*1.5: break
                x,y=xn,yn
        it=0
        while np.max(np.abs(y))>=1e-9 and it<10:
            J=np.empty((6,6)); h=1e-6; y0=y.copy()
            for j in range(6):
                xp=x.copy();xp[j]+=h;J[:,j]=(fn(xp)-y0)/h
            x=x-np.linalg.solve(J,y0);y=fn(x);self.tgf_J=J;it+=1
        err,st=self.evaluate(model,x); st['tgf_error']=np.max(np.abs(err));st['tgf_success']=st['tgf_error']<1e-9;return st


## Simulation helpers

Copied helper functions plus a grouped-parameter interface for GSA.

In [ ]:
def state_row(model,st):
    counts=np.array([n['count'] for n in model['nephrons']]); v=st['vascular']; mo=st['med_out']; urine=mo.cd_out
    gfr=float(np.sum(counts*v['gfr'])); netpf=float(np.sum(counts*v['bf'])*(1-model['hct']))
    row={'PA':model['p_art'],'GFR':gfr,'FF':gfr/netpf,'UTFP':gfr/urine[0],
         'PP_OSM':1.82*st['med_soln'][-2]+st['med_soln'][-1]+model['med_c_nrs'][-1],
         'PP_CNA':st['med_soln'][-2],'UR_OSM':mosm(urine),'UR_CNA':c_na(urine),
         'UR_JNA':urine[1],'UR_FR':urine[0],'PINT':v['p_int'],
         'TGF_ERROR':st['tgf_error'],'MED_ERROR':st['med_error'],'GFR_ERROR':v['gerr']}
    for i,z in enumerate(v['gfr']): row[f'SNGFR{i}']=z
    for i,z in enumerate(st['rtgf']): row[f'RTGF{i}']=z
    return row

def run_series(model_key,pressures=PRESSURES,verbose=True):
    model=deepcopy(EMBEDDED_MODELS[model_key]); solver=WholeKidneySolver(); rows=[]; states={}; prev=[]
    schedule=list(map(float,pressures))
    # Retain 1 mmHg continuation across the steep 90--110 mmHg transition;
    # outside it the requested output grid is already stable and exact.
    bridge=list(np.arange(min(110.,schedule[0]),schedule[-1]-.1,-1.))
    internal=sorted(set(schedule+bridge),reverse=True)
    for k,p in enumerate(internal):
        model['p_art']=p
        if len(prev)>=2:
            p0,s0=prev[-2];p1,s1=prev[-1]; fac=(p-p1)/(p1-p0)
            rt=s1['rtgf']+fac*(s1['rtgf']-s0['rtgf'])
            solver.med.x=s1['med_soln']+fac*(s1['med_soln']-s0['med_soln'])
            solver.gfr_z=s1['vascular']['z']+fac*(s1['vascular']['z']-s0['vascular']['z'])
        elif prev: rt=prev[-1][1]['rtgf']
        else: rt=None
        tic=time.time(); st=solver.solve(model,rt); prev.append((p,st)); prev=prev[-2:]
        if p in schedule:
            rows.append(state_row(model,st)); states[p]=st
            if verbose: print(f'{model_key:9s} PA={p:5.1f} GFR={rows[-1]["GFR"]:8.3f} urine={rows[-1]["UR_FR"]:9.4f} ({time.time()-tic:.1f}s)')
    return pd.DataFrame(rows),states

def profile_table(model,st):
    mo=st['med_out']; rows=[]
    def add(name,vessel,x,f,base):
        rows.append({'pa':model['p_art'],'name':name,'vessel':vessel,'x':x,'flow':f[0],
          'cna':c_na(f),'cur':c_urea(f),'mosm':mosm(f),'fdv':f[0]/base[0],
          'fdna':f[1]/base[1],'fdur':f[2]/base[2]})
    for i,n in enumerate(model['nephrons']):
        base=st['gflows'][i]
        for x,f in zip(np.linspace(0,n['dlh']['length'],len(mo.dlh_f[i])),mo.dlh_f[i]): add(f'dlh{i}','dlh',x,f,base)
        for x,f in zip(np.linspace(n['alh']['length'],0,len(mo.alh_f[i])),mo.alh_f[i]): add(f'alh{i}','alh',x,f,base)
        cort_alh=n['cort_alh'][1]['length'] if n['cort_alh'][0]=='Some' else 0.
        for j,f in enumerate(mo.dct_f[i]):
            along=j*n['cort_dx']; x=-along
            if along<=cort_alh+1e-12: add(f'alh{i}','alh',x,f,base)
            if along>=cort_alh-1e-12: add(f'dct{i}','dct',x,f,base)
    counts=np.array([n['count'] for n in model['nephrons']])
    cbase=np.sum([counts[i]*st['gflows'][i] for i in range(len(counts))],axis=0)
    for x,f in zip(np.linspace(0,.7,len(mo.cd_f)),mo.cd_f): add('cd0','cd',x,f,cbase)
    for i,z in enumerate(mo.istm): rows.append({'pa':model['p_art'],'name':'istm','vessel':'istm','x':(i+.5)*.01,'flow':np.nan,'cna':z[0],'cur':z[1],'mosm':1.82*z[0]+z[1]+z[3],'fdv':np.nan,'fdna':np.nan,'fdur':np.nan})
    return pd.DataFrame(rows)


In [ ]:
def load_parameter_table(path=PARAMETER_TABLE_PATH, sheet_name=PARAMETER_SHEET):
    df = read_parameter_table_file(path, sheet_name=sheet_name)
    required = {
        "input_index",
        "group_id",
        "member_count",
        "members",
        "baseline",
        "lower_bound",
        "upper_bound",
        "unit",
        "category",
        "oat_recommendation",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Parameter table is missing required columns: {missing}")
    df = df.copy()
    df["input_index"] = df["input_index"].astype(int)
    for col in ["baseline", "lower_bound", "upper_bound"]:
        df[col] = pd.to_numeric(df[col], errors="raise")
    if len(df) != 143:
        raise ValueError(f"Expected exactly 143 grouped parameters, found {len(df)}")
    if df["group_id"].duplicated().any():
        dup = df.loc[df["group_id"].duplicated(), "group_id"].tolist()
        raise ValueError(f"Duplicate group_id values: {dup[:5]}")
    return df.sort_values("input_index").reset_index(drop=True)


def split_members(members):
    if pd.isna(members):
        return []
    return [m.strip() for m in str(members).split(";") if m.strip()]


def _member_root(model, member_path):
    parts = member_path.split(".")
    if parts[0].startswith("N") and len(parts[0]) == 3 and parts[0][1:].isdigit():
        ix = int(parts[0][1:]) - 1
        obj = model["nephrons"][ix]
        parts = parts[1:]
    else:
        obj = model
    normalized = []
    skip = False
    for i, token in enumerate(parts):
        if skip:
            skip = False
            continue
        normalized.append(token)
        if token in {"Permeable", "Active"} and i + 1 < len(parts) and parts[i + 1] == "1":
            skip = True
    return obj, normalized


def _descend(obj, token):
    if isinstance(obj, dict):
        return obj[token]
    if isinstance(obj, list):
        if token == "Some":
            if obj[0] != "Some":
                raise KeyError("Expected a Some option wrapper")
            return obj[1]
        if token in {"Permeable", "Active"}:
            if obj[0] != token:
                raise KeyError(f"Expected transport wrapper {token}")
            return obj[1]
        if token == "piece_value":
            return obj[1]
        if token in {"Const", "Linear", "Exp"}:
            if obj[0] != token:
                raise KeyError(f"Expected range generator {token}")
            return obj[1]
        if token.isdigit():
            return obj[int(token)]
    raise KeyError(f"Cannot descend through token {token!r} in object type {type(obj).__name__}")


def _get_final(obj, token):
    if isinstance(obj, dict):
        value = obj[token]
        if isinstance(value, list) and len(value) == 2 and value[0] == "Some":
            return value[1]
        return value
    if isinstance(obj, list):
        if token == "Const":
            if obj[0] != "Const":
                raise KeyError("Expected Const range generator")
            return obj[1]
        if token == "start":
            return obj[0]
        if token == "end":
            return obj[1]
        if token == "exponent":
            return obj[2]
        if token.isdigit():
            return obj[int(token)]
    raise KeyError(f"Cannot read final token {token!r}")


def _set_final(obj, token, value):
    value = float(value)
    if isinstance(obj, dict):
        current = obj[token]
        if isinstance(current, list) and len(current) == 2 and current[0] == "Some":
            current[1] = value
        else:
            obj[token] = value
        return
    if isinstance(obj, list):
        if token == "Const":
            if obj[0] != "Const":
                raise KeyError("Expected Const range generator")
            obj[1] = value
            return
        if token == "start":
            obj[0] = value
            return
        if token == "end":
            obj[1] = value
            return
        if token == "exponent":
            obj[2] = value
            return
        if token.isdigit():
            obj[int(token)] = value
            return
    raise KeyError(f"Cannot set final token {token!r}")


def get_member_value(model, member_path):
    obj, parts = _member_root(model, member_path)
    for token in parts[:-1]:
        obj = _descend(obj, token)
    return _get_final(obj, parts[-1])


def set_member_value(model, member_path, value):
    obj, parts = _member_root(model, member_path)
    for token in parts[:-1]:
        obj = _descend(obj, token)
    _set_final(obj, parts[-1], value)


def validate_parameter_table_against_model(parameter_table=None, atol=1e-12, rtol=1e-9):
    if parameter_table is None:
        parameter_table = load_parameter_table()
    model = deepcopy(EMBEDDED_MODELS[MODEL_KEY])
    rows = []
    for _, row in parameter_table.iterrows():
        members = split_members(row["members"])
        member_values = []
        errors = []
        for member in members:
            try:
                member_values.append(float(get_member_value(model, member)))
            except Exception as exc:
                errors.append(f"{member}: {type(exc).__name__}: {exc}")
        expected = float(row["baseline"])
        ok = (not errors) and len(member_values) == int(row["member_count"]) and all(
            np.isclose(v, expected, atol=atol, rtol=rtol) for v in member_values
        )
        rows.append({
            "input_index": row["input_index"],
            "group_id": row["group_id"],
            "member_count_declared": int(row["member_count"]),
            "member_count_found": len(member_values),
            "baseline_expected": expected,
            "min_member_baseline": min(member_values) if member_values else np.nan,
            "max_member_baseline": max(member_values) if member_values else np.nan,
            "recognized_by_model": not errors,
            "baseline_matches_model": ok,
            "validation_errors": " | ".join(errors),
        })
    return pd.DataFrame(rows)


def apply_grouped_parameter(model, parameter_row, value):
    for member in split_members(parameter_row["members"]):
        set_member_value(model, member, value)
    return model


def build_model_from_parameter_values(parameter_values=None, parameter_table=None):
    if parameter_table is None:
        parameter_table = load_parameter_table()
    model = deepcopy(EMBEDDED_MODELS[MODEL_KEY])
    if parameter_values:
        table_by_group = parameter_table.set_index("group_id", drop=False)
        table_by_index = parameter_table.set_index("input_index", drop=False)
        for key, value in parameter_values.items():
            if key in table_by_group.index:
                row = table_by_group.loc[key]
            elif isinstance(key, int) and key in table_by_index.index:
                row = table_by_index.loc[key]
            else:
                raise KeyError(f"Unknown grouped parameter: {key}")
            apply_grouped_parameter(model, row, value)
    return model


def run_model_instance(model, return_details=False):
    solver = WholeKidneySolver()
    started = time.time()
    state = solver.solve(model)
    runtime_seconds = time.time() - started
    row = state_row(model, state)
    row["runtime_seconds"] = runtime_seconds
    if return_details:
        return row, state
    return row


def run_forward_model(parameter_values=None, parameter_table=None, return_details=False):
    model = build_model_from_parameter_values(parameter_values, parameter_table)
    return run_model_instance(model, return_details=return_details)


def moss_forward_gfr(parameter_values=None, parameter_table=None):
    """Run the grouped-parameter MOSS forward model and return only GFR.

    parameter_values can be keyed by group_id, for example
    {"p_art|baseline=140": 120.0}, or by integer input_index.
    """
    result = run_forward_model(
        parameter_values=parameter_values,
        parameter_table=parameter_table,
        return_details=False,
    )
    return float(result[PRIMARY_OUTPUT])


GSA_PARAMETER_TABLE = load_parameter_table()
PARAMETER_VALIDATION = validate_parameter_table_against_model(GSA_PARAMETER_TABLE)
print("Grouped parameters loaded:", len(GSA_PARAMETER_TABLE))
print("All 143 recognized and baseline-matched:", bool(PARAMETER_VALIDATION["baseline_matches_model"].all()))


In [ ]:
baseline_check = run_forward_model(return_details=False)
baseline_check